# Детектор ТС: дообучение YOLO11 на UA-DETRAC (Google Colab)

Этап 2 порядка разработки АИС из ТЗ: "Разработка и обучение модели
детекции — дообучение модели семейства YOLOv11 на задаче обнаружения ТС
на изображении".

Детектор находит единственный класс **vehicle** (car/bus/van/truck
объединены в один класс — так задумано архитектурой системы: детектор
только находит "где на кадре есть транспорт", а классификация марки/типа
в задачи этой модели не входит).

**Ключевая идея этого ноутбука** (важно, если у вас ограничено место на
Google Drive): полный UA-DETRAC (~10 ГБ на Kaggle) скачивается на
ВРЕМЕННЫЙ локальный диск самой машины Colab (`/content`, обычно ~100+ ГБ
своб. места, обнуляется между сессиями), а НЕ на Google Drive. Из него
берётся только небольшое ПОДМНОЖЕСТВО последовательностей и кадров,
конвертируется в компактный YOLO-датасет (сотни МБ), и именно ЭТОТ
маленький результат кешируется на Drive — сырые 10 ГБ на Drive никогда
не попадают.

**Перед запуском:**

1. `Runtime -> Change runtime type -> GPU`.
2. Получите API-токен Kaggle: `kaggle.com` -> `Settings` -> `API` ->
   `Create New Token` -> скачается `kaggle.json`.
3. Скопируйте "слаг" датасета из его URL на Kaggle:
   `kaggle.com/datasets/ВЛАДЕЛЕЦ/НАЗВАНИЕ` -> слаг `ВЛАДЕЛЕЦ/НАЗВАНИЕ`.
   Подставьте его в `KAGGLE_DATASET_SLUG` на Шаге 5.


## Шаг 1. Проверка GPU

In [1]:
!nvidia-smi

Sat Sep 19 06:56:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Шаг 2. Установка зависимостей

In [2]:
!pip install -q ultralytics pyyaml onnx onnxslim kaggle
import ultralytics
ultralytics.checks()

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.4/112.6 GB disk)


## Шаг 3. Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/VehicleReID'  # тот же проект, что и для ReID

Mounted at /content/drive


## Шаг 4. Код проекта: конвертер датасета и обучающие скрипты

Записываем те же модули, что использовались бы в финальной сборке
проекта (`ml/detection/prepare_dataset.py`, `train_detector.py`,
`export_detector.py`) — идентичный код и в Colab, и в проекте, чтобы не
было расхождений между тем, что обучалось, и тем, что попадёт в
приложение.

In [4]:
import os
os.makedirs('/content/ml/detection', exist_ok=True)

In [5]:
%%writefile /content/ml/detection/prepare_dataset.py
"""
Подготовка датасета UA-DETRAC для дообучения YOLO11 на задаче детекции
единого класса "vehicle" (согласно архитектуре системы - детектор не
различает car/bus/van/truck, а находит любое транспортное средство).

Почему UA-DETRAC, а не COCO: UA-DETRAC - это именно кадры со стационарных
камер городского видеонаблюдения (Пекин/Тяньцзинь), что напрямую
соответствует домену эксплуатации системы согласно ТЗ ("камеры
видеонаблюдения, установленные в городской среде"). YOLO11 уже
предобучен на COCO (там есть car/truck/bus), поэтому дообучение именно
на CCTV-специфичных кадрах даёт реальную пользу, а не просто повторяет
то, что модель уже знает.

Ожидаемая структура исходных файлов UA-DETRAC (нативный релиз датасета,
https://detrac-db.rit.albany.edu/download):

    UA-DETRAC/
        Insight-MVT_Annotation_Train/
            MVI_20011/
                img00001.jpg
                img00002.jpg
                ...
            MVI_20012/
                ...
        DETRAC-Train-Annotations-XML/
            MVI_20011.xml
            MVI_20012.xml
            ...

Формат XML-аннотации (подтверждённая структура нативного релиза):

    <sequence name="MVI_20011">
      <frame num="1">
        <target_list>
          <target id="1">
            <box left="..." top="..." width="..." height="..."/>
            <attribute vehicle_type="car" .../>
          </target>
        </target_list>
      </frame>
      ...
    </sequence>

Скрипт:
    1. Парсит все XML-файлы аннотаций.
    2. Конвертирует box (left, top, width, height) в формат YOLO
       (class x_center y_center width height, все значения нормированы
       на размер кадра в диапазоне [0,1]).
    3. Все типы vehicle_type (car, bus, van, others) сливаются в ОДИН
       класс "vehicle" (индекс 0), согласно архитектуре системы.
    4. Опционально отбрасывает сильно перекрытые/обрезанные боксы
       (occlusion_ratio / truncation_ratio выше порога) - такие объекты
       обычно вредят обучению детектора, а не помогают.
    5. Разбивает ПОСЛЕДОВАТЕЛЬНОСТИ (не отдельные кадры!) на train/val,
       чтобы избежать утечки данных - соседние кадры одной
       последовательности почти идентичны, и если раскидать их по
       train/val случайно, оценка качества будет необъективно завышена.
    6. Копирует/симлинкает изображения в стандартную YOLO-структуру
       (images/train, images/val, labels/train, labels/val) и пишет
       data.yaml для передачи в ultralytics.

Пример запуска:

    python ml/detection/prepare_dataset.py \
        --detrac-root /path/to/UA-DETRAC \
        --output data/detection_yolo \
        --val-fraction 0.15
"""

from __future__ import annotations

import argparse
import os
import random
import shutil
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import List, Tuple

import yaml


VEHICLE_CLASS_ID = 0  # единственный класс - "vehicle"


@dataclass
class YoloBox:
    class_id: int
    x_center: float
    y_center: float
    width: float
    height: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Конвертация UA-DETRAC в формат YOLO (единый класс vehicle)"
    )
    parser.add_argument("--detrac-root", type=str, required=True,
                         help="Корень UA-DETRAC (содержит Insight-MVT_Annotation_Train/ и DETRAC-Train-Annotations-XML/)")
    parser.add_argument("--images-subdir", type=str, default="Insight-MVT_Annotation_Train")
    parser.add_argument("--annotations-subdir", type=str, default="DETRAC-Train-Annotations-XML")
    parser.add_argument("--output", type=str, required=True,
                         help="Куда записать датасет в формате YOLO")
    parser.add_argument("--val-fraction", type=float, default=0.15,
                         help="Доля ПОСЛЕДОВАТЕЛЬНОСТЕЙ (не кадров), уходящих в val")
    parser.add_argument("--max-sequences", type=int, default=None,
                         help="Ограничить число обрабатываемых последовательностей "
                              "(случайный выбор с учётом --seed). Полезно, когда "
                              "полный датасет слишком велик для доступного места/времени "
                              "(например, при обучении в Google Colab) - можно взять, "
                              "скажем, 15 последовательностей из 100 вместо всех. "
                              "По умолчанию используются ВСЕ найденные последовательности.")
    parser.add_argument("--max-occlusion-ratio", type=float, default=1.0,
                         help="Отбрасывать боксы с occlusion_ratio выше порога (1.0 = не фильтровать)")
    parser.add_argument("--max-truncation-ratio", type=float, default=1.0,
                         help="Отбрасывать боксы с truncation_ratio выше порога (1.0 = не фильтровать)")
    parser.add_argument("--link-mode", choices=["copy", "symlink"], default="symlink",
                         help="copy - дублировать файлы изображений; symlink - создавать символьные "
                              "ссылки (быстрее и не расходует лишнее место на диске, но требует "
                              "прав на symlink - на Windows без прав администратора может не сработать)")
    parser.add_argument("--frame-stride", type=int, default=1,
                         help="Брать только каждый N-й кадр последовательности (по номеру "
                              "кадра). UA-DETRAC снят с 25 fps, и соседние кадры почти "
                              "дублируют друг друга - для обучения детектора это в основном "
                              "избыточность, а не полезное разнообразие. --frame-stride 5 "
                              "снижает объём датасета впятеро почти без потери качества "
                              "обучения, что особенно полезно при ограниченном месте/времени "
                              "(например, в Google Colab). По умолчанию 1 = берутся все кадры.")
    parser.add_argument("--seed", type=int, default=42)

    return parser.parse_args()


def _parse_sequence_xml(
    xml_path: str,
    max_occlusion_ratio: float,
    max_truncation_ratio: float,
) -> dict:
    """
    Парсит один XML-файл аннотаций UA-DETRAC (одна последовательность).

    Returns:
        {frame_num: [YoloBox без нормировки на размер кадра (в пикселях,
         но уже в формате left/top/width/height -> x_center/y_center/w/h)]}
        Нормировка на размер изображения делается отдельно, т.к. XML не
        всегда содержит явный размер кадра - берём его из самого файла
        изображения при записи.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    frames = {}
    for frame_tag in root.findall("frame"):
        frame_num = int(frame_tag.attrib["num"])
        boxes_px = []  # (left, top, width, height) в пикселях

        target_list = frame_tag.find("target_list")
        if target_list is None:
            frames[frame_num] = boxes_px
            continue

        for target in target_list.findall("target"):
            box_tag = target.find("box")
            if box_tag is None:
                continue

            attribute_tag = target.find("attribute")
            occlusion_ratio = 0.0
            truncation_ratio = 0.0
            if attribute_tag is not None:
                occlusion_ratio = float(attribute_tag.attrib.get("occlusion_ratio", 0.0) or 0.0)
                truncation_ratio = float(attribute_tag.attrib.get("truncation_ratio", 0.0) or 0.0)

            if occlusion_ratio > max_occlusion_ratio:
                continue
            if truncation_ratio > max_truncation_ratio:
                continue

            left = float(box_tag.attrib["left"])
            top = float(box_tag.attrib["top"])
            width = float(box_tag.attrib["width"])
            height = float(box_tag.attrib["height"])
            boxes_px.append((left, top, width, height))

        frames[frame_num] = boxes_px

    return frames


def _boxes_to_yolo_lines(boxes_px: List[Tuple[float, float, float, float]], img_w: int, img_h: int) -> List[str]:
    lines = []
    for left, top, width, height in boxes_px:
        x_center = (left + width / 2.0) / img_w
        y_center = (top + height / 2.0) / img_h
        w_norm = width / img_w
        h_norm = height / img_h

        # Отбрасываем полностью вырожденные/вышедшие за кадр боксы.
        if w_norm <= 0 or h_norm <= 0:
            continue
        x_center = min(max(x_center, 0.0), 1.0)
        y_center = min(max(y_center, 0.0), 1.0)
        w_norm = min(w_norm, 1.0)
        h_norm = min(h_norm, 1.0)

        lines.append(f"{VEHICLE_CLASS_ID} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
    return lines


def _link_or_copy(src: str, dst: str, mode: str) -> None:
    if os.path.exists(dst):
        return
    if mode == "symlink":
        try:
            os.symlink(os.path.abspath(src), dst)
            return
        except OSError:
            pass  # на некоторых системах (Windows без прав) symlink не создать - копируем
    shutil.copy2(src, dst)


def main() -> None:
    args = parse_args()
    random.seed(args.seed)

    images_root = os.path.join(args.detrac_root, args.images_subdir)
    annotations_root = os.path.join(args.detrac_root, args.annotations_subdir)

    if not os.path.isdir(images_root):
        raise FileNotFoundError(f"Не найдена папка изображений: {images_root}")
    if not os.path.isdir(annotations_root):
        raise FileNotFoundError(f"Не найдена папка аннотаций: {annotations_root}")

    xml_files = sorted(f for f in os.listdir(annotations_root) if f.endswith(".xml"))
    if not xml_files:
        raise FileNotFoundError(f"В {annotations_root} не найдено .xml файлов")

    sequence_names = [os.path.splitext(f)[0] for f in xml_files]
    random.shuffle(sequence_names)

    if args.max_sequences is not None and args.max_sequences < len(sequence_names):
        print(
            f"Ограничиваю число последовательностей: {len(sequence_names)} -> "
            f"{args.max_sequences} (случайный выбор, seed={args.seed})"
        )
        sequence_names = sequence_names[: args.max_sequences]

    num_val = max(1, int(round(len(sequence_names) * args.val_fraction)))
    val_sequences = set(sequence_names[:num_val])
    train_sequences = set(sequence_names[num_val:])

    print(f"Всего последовательностей: {len(sequence_names)}")
    print(f"  train: {len(train_sequences)}, val: {len(val_sequences)}")
    print(
        "  (разбиение по ПОСЛЕДОВАТЕЛЬНОСТЯМ целиком, а не по отдельным "
        "кадрам - иначе соседние кадры одной сцены попадут и в train, и "
        "в val, завышая оценку качества)"
    )

    for split in ("train", "val"):
        os.makedirs(os.path.join(args.output, "images", split), exist_ok=True)
        os.makedirs(os.path.join(args.output, "labels", split), exist_ok=True)

    # Отложенный импорт, чтобы скрипт можно было запустить даже без Pillow
    # установленного глобально (проверка происходит только здесь).
    from PIL import Image

    stats = {"train": {"images": 0, "boxes": 0}, "val": {"images": 0, "boxes": 0}}

    for seq_name in sequence_names:
        split = "val" if seq_name in val_sequences else "train"

        xml_path = os.path.join(annotations_root, f"{seq_name}.xml")
        seq_image_dir = os.path.join(images_root, seq_name)
        if not os.path.isdir(seq_image_dir):
            print(f"  ПРЕДУПРЕЖДЕНИЕ: нет папки изображений для {seq_name}, пропускаю")
            continue

        frames = _parse_sequence_xml(xml_path, args.max_occlusion_ratio, args.max_truncation_ratio)

        for frame_num, boxes_px in frames.items():
            if (frame_num - 1) % args.frame_stride != 0:
                continue

            img_filename = f"img{frame_num:05d}.jpg"
            src_img_path = os.path.join(seq_image_dir, img_filename)
            if not os.path.isfile(src_img_path):
                continue

            with Image.open(src_img_path) as img:
                img_w, img_h = img.size

            yolo_lines = _boxes_to_yolo_lines(boxes_px, img_w, img_h)

            # Уникальное имя, т.к. img00001.jpg повторяется в каждой
            # последовательности - добавляем префикс seq_name.
            unique_name = f"{seq_name}_{img_filename}"
            dst_img_path = os.path.join(args.output, "images", split, unique_name)
            dst_label_path = os.path.join(
                args.output, "labels", split, os.path.splitext(unique_name)[0] + ".txt"
            )

            _link_or_copy(src_img_path, dst_img_path, args.link_mode)
            with open(dst_label_path, "w", encoding="utf-8") as f:
                f.write("\n".join(yolo_lines))
                if yolo_lines:
                    f.write("\n")

            stats[split]["images"] += 1
            stats[split]["boxes"] += len(yolo_lines)

    data_yaml = {
        "path": os.path.abspath(args.output),
        "train": "images/train",
        "val": "images/val",
        "names": {VEHICLE_CLASS_ID: "vehicle"},
    }
    data_yaml_path = os.path.join(args.output, "data.yaml")
    with open(data_yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

    print()
    print("Готово.")
    for split in ("train", "val"):
        print(f"  {split}: {stats[split]['images']} изображений, {stats[split]['boxes']} боксов")
    print(f"data.yaml записан в: {data_yaml_path}")


if __name__ == "__main__":
    main()


Writing /content/ml/detection/prepare_dataset.py


In [6]:
%%writefile /content/ml/detection/train_detector.py
"""
Дообучение YOLO11 (YOLO11s / YOLO11m) на задаче детекции единого класса
"vehicle", согласно этапу 2 порядка разработки АИС из ТЗ:
"Разработка и обучение модели детекции — дообучение модели семейства
YOLOv11 на задаче обнаружения ТС на изображении".

Модель стартует с ImageNet/COCO-претрейна ultralytics (YOLO11 уже умеет
находить car/truck/bus "из коробки") и дообучается на CCTV-специфичных
кадрах UA-DETRAC, подготовленных ml/detection/prepare_dataset.py.

Пример запуска:

    python ml/detection/train_detector.py \
        --data data/detection_yolo/data.yaml \
        --model yolo11s.pt \
        --epochs 50 \
        --imgsz 640 \
        --batch 16 \
        --project ml/detection/runs \
        --name vehicle_detector

Возобновление после прерывания (ultralytics хранит last.pt в
<project>/<name>/weights/last.pt и сам восстанавливает состояние
оптимизатора/эпохи):

    python ml/detection/train_detector.py \
        --resume ml/detection/runs/vehicle_detector/weights/last.pt
"""

from __future__ import annotations

import argparse
import os

from ultralytics import YOLO


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Дообучение YOLO11 на детекции vehicle")

    parser.add_argument("--data", type=str,
                         help="Путь к data.yaml (из prepare_dataset.py). "
                              "Не требуется при --resume.")
    parser.add_argument("--model", type=str, default="yolo11s.pt",
                         help="Стартовый чекпоинт: yolo11n.pt (самый лёгкий) / "
                              "yolo11s.pt / yolo11m.pt (по архитектуре проекта - "
                              "s или m). Скачивается автоматически ultralytics "
                              "при первом запуске.")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--patience", type=int, default=15,
                         help="Остановка, если mAP не растёт N эпох подряд (early stopping)")
    parser.add_argument("--device", type=str, default=None,
                         help="'0' для первого GPU, 'cpu' для CPU. По умолчанию "
                              "ultralytics сам выбирает GPU, если доступен.")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--project", type=str, default="ml/detection/runs",
                         help="Куда сохранять результаты обучения (веса, графики, логи)")
    parser.add_argument("--name", type=str, default="vehicle_detector",
                         help="Имя конкретного запуска (подпапка внутри --project)")
    parser.add_argument("--resume", type=str, default=None,
                         help="Путь к last.pt для возобновления прерванного обучения "
                              "(если указан, --data/--model/--project/--name не нужны - "
                              "ultralytics берёт их из сохранённой конфигурации запуска)")
    parser.add_argument("--seed", type=int, default=42)

    args = parser.parse_args()
    if args.resume is None and args.data is None:
        parser.error("Нужно указать --data (для нового обучения) или --resume (для продолжения)")
    return args


def main() -> None:
    args = parse_args()

    if args.resume:
        print(f"Возобновляю обучение из: {args.resume}")
        model = YOLO(args.resume)
        model.train(resume=True)
        return

    if not os.path.isfile(args.data):
        raise FileNotFoundError(
            f"Не найден {args.data}. Сначала запустите "
            f"ml/detection/prepare_dataset.py, чтобы получить data.yaml."
        )

    print(f"Загружаю стартовые веса: {args.model}")
    model = YOLO(args.model)

    train_kwargs = dict(
        data=args.data,
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        patience=args.patience,
        workers=args.workers,
        project=args.project,
        name=args.name,
        seed=args.seed,
        # Единственный класс "vehicle" - боксы разного типа транспорта
        # уже слиты в один класс на этапе prepare_dataset.py, здесь
        # никаких дополнительных настроек под multi-class не нужно.
    )
    if args.device is not None:
        train_kwargs["device"] = args.device

    results = model.train(**train_kwargs)

    best_path = os.path.join(args.project, args.name, "weights", "best.pt")
    print()
    print("Обучение завершено.")
    print(f"Лучший чекпоинт: {best_path}")
    print(f"Метрики финального запуска: {results}")


if __name__ == "__main__":
    main()


Writing /content/ml/detection/train_detector.py


In [7]:
%%writefile /content/ml/detection/export_detector.py
"""
Экспорт обученной YOLO11-модели детекции в ONNX, согласно этапу 4
порядка разработки АИС из ТЗ: "Экспорт и оптимизация моделей —
подготовка обученных моделей к использованию в составе АИС".

Зачем ONNX, а не просто .pt: ТЗ явно требует (п. 1.3.2.3), чтобы
серверная часть могла работать "как с использованием GPU, так и на CPU
при отсутствии ускорителя". Экспорт в ONNX даёт переносимый формат,
который ultralytics и onnxruntime одинаково умеют запускать на обоих
устройствах, и который проще версионировать/деплоить в Docker-образе
сервера, не таская за собой полный вес обучающего фреймворка.

Пример запуска:

    python ml/detection/export_detector.py \
        --checkpoint ml/detection/runs/vehicle_detector/weights/best.pt \
        --output ml/detection/exported \
        --imgsz 640
"""

from __future__ import annotations

import argparse
import os
import shutil

from ultralytics import YOLO


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Экспорт YOLO11 детектора в ONNX")

    parser.add_argument("--checkpoint", type=str, required=True,
                         help="Путь к обученному best.pt")
    parser.add_argument("--output", type=str, required=True,
                         help="Папка, куда положить итоговый .onnx файл")
    parser.add_argument("--imgsz", type=int, default=640,
                         help="Размер входа, ЗАФИКСИРОВАННЫЙ в графе ONNX. "
                              "Должен совпадать с --imgsz, использованным при "
                              "обучении (train_detector.py), иначе точность "
                              "детекции на инференсе может ухудшиться.")
    parser.add_argument("--dynamic", action="store_true",
                         help="Разрешить переменный размер батча/входа в ONNX-графе "
                              "(немного медленнее, но гибче для разных клиентов)")
    parser.add_argument("--simplify", action="store_true", default=True,
                         help="Упростить ONNX-граф через onnxsim (обычно ускоряет "
                              "инференс без потери точности)")
    parser.add_argument("--opset", type=int, default=17)

    return parser.parse_args()


def main() -> None:
    args = parse_args()

    if not os.path.isfile(args.checkpoint):
        raise FileNotFoundError(f"Не найден чекпоинт: {args.checkpoint}")

    os.makedirs(args.output, exist_ok=True)

    print(f"Загружаю чекпоинт: {args.checkpoint}")
    model = YOLO(args.checkpoint)

    print(f"Экспортирую в ONNX (imgsz={args.imgsz}, opset={args.opset})...")
    exported_path = model.export(
        format="onnx",
        imgsz=args.imgsz,
        dynamic=args.dynamic,
        simplify=args.simplify,
        opset=args.opset,
    )

    final_path = os.path.join(args.output, "vehicle_detector.onnx")
    shutil.move(str(exported_path), final_path)

    print()
    print("Готово.")
    print(f"ONNX-модель сохранена: {final_path}")
    print(
        "Используйте этот путь в backend/app/services/detector.py "
        "(YOLO(path_to_onnx) работает точно так же, как YOLO(path_to_pt), "
        "ultralytics сам определяет формат по расширению файла)."
    )


if __name__ == "__main__":
    main()


Writing /content/ml/detection/export_detector.py


## Шаг 5. Получение и обрезка данных

Логика:

1. Если на Drive уже лежит закешированный (маленький, уже обрезанный)
   YOLO-датасет с прошлого запуска — используем его, и сырые данные с
   Kaggle заново вообще не скачиваем.
2. Если кеша ещё нет — скачиваем полный архив с Kaggle на локальный
   диск Colab (`/content`, НЕ Drive), конвертируем только
   `MAX_SEQUENCES` последовательностей с прореживанием кадров
   `FRAME_STRIDE`, и полученный маленький результат сохраняем на Drive
   для будущих запусков.

Настройте под свои ограничения по месту/времени:

In [8]:
KAGGLE_DATASET_SLUG = "bratjay/ua-detrac-orig"
KAGGLE_API_TOKEN = "KGAT_368670e09bf547a86d1c197c3af94854"

MAX_SEQUENCES = 15     # сколько последовательностей ИЗ 100 реально использовать
FRAME_STRIDE = 5       # брать каждый N-й кадр внутри последовательности (25 fps -> кадры почти дублируются)
VAL_FRACTION = 0.15
MAX_OCCLUSION_RATIO = 0.8

In [12]:
import os

CACHED_YOLO_ZIP = f'{DRIVE_PROJECT_ROOT}/data/detection_yolo_subset.zip'
LOCAL_YOLO_DATASET = '/content/data/detection_yolo'

os.makedirs(f'{DRIVE_PROJECT_ROOT}/data', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

if os.path.isfile(CACHED_YOLO_ZIP):
    print('Найден закешированный обработанный датасет на Drive - использую его.')
    print('(сырые ~10 ГБ с Kaggle заново скачивать не нужно)')
    !unzip -q "{CACHED_YOLO_ZIP}" -d /content/data

else:
    print('Кеша на Drive нет - скачиваю сырые данные с Kaggle на ЛОКАЛЬНЫЙ диск Colab...')

    # --- Аутентификация в Kaggle API (новый формат: единый токен) ---
    # Kaggle недавно сменил формат выдачи credentials: вместо файла
    # kaggle.json (username+key) теперь выдаётся один токен через
    # 'Generate New Token'. Правильный способ его использовать -
    # переменная окружения KAGGLE_API_TOKEN (или файл
    # ~/.kaggle/access_token) - без создания kaggle.json вручную.
    #
    # Если у вас всё же СТАРЫЙ kaggle.json (username+key) - он тоже
    # продолжает работать, тогда вместо строки ниже нужно загрузить
    # сам файл и положить его в ~/.kaggle/kaggle.json, как раньше.
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

    # --- Скачивание на ЛОКАЛЬНЫЙ диск (/content), НЕ на Drive ---
    LOCAL_RAW_ROOT = '/content/data/UA-DETRAC_raw'
    os.makedirs(LOCAL_RAW_ROOT, exist_ok=True)
    !kaggle datasets download -d "{KAGGLE_DATASET_SLUG}" --unzip -p "{LOCAL_RAW_ROOT}"

    # --- Автопоиск папок с кадрами и аннотациями внутри скачанного архива ---
    # Разные зеркала на Kaggle по-разному называют и вкладывают папки
    # (иногда добавляют лишний уровень, иногда переименовывают
    # Insight-MVT_Annotation_Train во что-то своё). Поэтому ищем не по
    # ТОЧНОМУ имени папки, а по её СОДЕРЖИМОМУ: папку с кадрами - по
    # наличию подпапок MVI_*, папку с аннотациями - по наличию файлов
    # MVI_*.xml (с предпочтением пути, содержащего "Train", чтобы не
    # перепутать с DETRAC-Test-Annotations-XML).
    def find_dir_with_mvi_subdirs(search_root):
        best_dir, best_count = None, 0
        for dirpath, dirnames, _ in os.walk(search_root):
            mvi_dirs = [d for d in dirnames if d.startswith('MVI_')]
            if len(mvi_dirs) > best_count:
                best_dir, best_count = dirpath, len(mvi_dirs)
        return best_dir if best_count >= 5 else None

    def find_dir_with_mvi_xml(search_root, prefer_keyword='Train'):
        candidates = []  # (path, count_of_xml_files)
        for dirpath, _, filenames in os.walk(search_root):
            xml_files = [f for f in filenames if f.startswith('MVI_') and f.endswith('.xml')]
            if len(xml_files) >= 5:
                candidates.append((dirpath, len(xml_files)))
        if not candidates:
            return None
        preferred = [c for c in candidates if prefer_keyword.lower() in c[0].lower()]
        pool = preferred if preferred else candidates
        return max(pool, key=lambda c: c[1])[0]

    IMAGES_DIR = find_dir_with_mvi_subdirs(LOCAL_RAW_ROOT)
    ANNOTATIONS_DIR = find_dir_with_mvi_xml(LOCAL_RAW_ROOT, prefer_keyword='Train')

    if IMAGES_DIR is None or ANNOTATIONS_DIR is None:
        print('НЕ УДАЛОСЬ автоматически найти папки с кадрами и/или аннотациями.')
        print('Содержимое архива для ручной проверки:')
        !find "{LOCAL_RAW_ROOT}" -maxdepth 4
        raise FileNotFoundError('Проверьте структуру датасета вручную и укажите пути явно.')

    print(f'Найдена папка с кадрами:     {IMAGES_DIR}')
    print(f'Найдена папка с аннотациями: {ANNOTATIONS_DIR}')

    # --- Конвертация в YOLO-формат с обрезкой по подмножеству/страйду ---
    # IMAGES_DIR/ANNOTATIONS_DIR передаются как АБСОЛЮТНЫЕ пути в
    # --images-subdir/--annotations-subdir: os.path.join игнорирует
    # --detrac-root, если подпуть уже абсолютный, поэтому значение
    # --detrac-root здесь не принципиально.
    !python /content/ml/detection/prepare_dataset.py \
        --detrac-root "{LOCAL_RAW_ROOT}" \
        --images-subdir "{IMAGES_DIR}" \
        --annotations-subdir "{ANNOTATIONS_DIR}" \
        --output "{LOCAL_YOLO_DATASET}" \
        --max-sequences {MAX_SEQUENCES} \
        --frame-stride {FRAME_STRIDE} \
        --val-fraction {VAL_FRACTION} \
        --max-occlusion-ratio {MAX_OCCLUSION_RATIO} \
        --link-mode copy

    # --- Кешируем маленький результат на Drive для будущих запусков ---
    print('Архивирую результат и сохраняю на Drive для последующих сессий...')
    !cd /content/data && zip -q -r detection_yolo_subset.zip detection_yolo
    !cp /content/data/detection_yolo_subset.zip "{CACHED_YOLO_ZIP}"
    print(f'Сохранено: {CACHED_YOLO_ZIP}')

    # Сырые данные (~10 ГБ) больше не нужны в этой сессии - освобождаем
    # локальный диск Colab (на Drive они и не попадали).
    import shutil
    shutil.rmtree(LOCAL_RAW_ROOT, ignore_errors=True)
    print('Временные сырые данные удалены с локального диска Colab.')

Кеша на Drive нет - скачиваю сырые данные с Kaggle на ЛОКАЛЬНЫЙ диск Colab...
Dataset URL: https://www.kaggle.com/datasets/bratjay/ua-detrac-orig
License(s): unknown
100% 9.23G/9.23G [01:29<00:00, 111MB/s]

Найдена папка с кадрами:     /content/data/UA-DETRAC_raw/DETRAC-Images/DETRAC-Images
Найдена папка с аннотациями: /content/data/UA-DETRAC_raw/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML
Ограничиваю число последовательностей: 60 -> 15 (случайный выбор, seed=42)
Всего последовательностей: 15
  train: 13, val: 2
  (разбиение по ПОСЛЕДОВАТЕЛЬНОСТЯМ целиком, а не по отдельным кадрам - иначе соседние кадры одной сцены попадут и в train, и в val, завышая оценку качества)

Готово.
  train: 3557 изображений, 34132 боксов
  val: 573 изображений, 3506 боксов
data.yaml записан в: /content/data/detection_yolo/data.yaml
Архивирую результат и сохраняю на Drive для последующих сессий...
Сохранено: /content/drive/MyDrive/VehicleReID/data/detection_yolo_subset.zip
Временные сырые данные

## Шаг 6. Обучение

Чекпоинты (`best.pt` / `last.pt`) пишутся **напрямую на Google Drive**
через `--project`. В отличие от ReID-обучения, здесь не нужна отдельная
ручная ротация чекпоинтов: ultralytics по умолчанию хранит только
`best.pt` и `last.pt` на запуск — компактно само по себе.

**Возобновление после разрыва сессии:** если `weights/last.pt` уже
существует по пути ниже, ultralytics сам восстановит состояние
оптимизатора и продолжит с прерванной эпохи.

In [13]:
DETECTOR_MODEL = 'yolo11s.pt'   # yolo11n.pt (легче/быстрее) или yolo11m.pt (точнее/тяжелее)
DETECTOR_PROJECT = f'{DRIVE_PROJECT_ROOT}/detection_runs'
DETECTOR_NAME = 'vehicle_detector'
DETECTOR_EPOCHS = 30   # лёгкое дообучение на небольшом подмножестве - не нужно 50-60 эпох
DETECTOR_IMGSZ = 640
DETECTOR_BATCH = 16

last_checkpoint = os.path.join(DETECTOR_PROJECT, DETECTOR_NAME, 'weights', 'last.pt')

if os.path.isfile(last_checkpoint):
    print(f'Найден чекпоинт для возобновления: {last_checkpoint}')
    !python /content/ml/detection/train_detector.py --resume "{last_checkpoint}"
else:
    print('Начинаю обучение с нуля.')
    !python /content/ml/detection/train_detector.py \
        --data {LOCAL_YOLO_DATASET}/data.yaml \
        --model {DETECTOR_MODEL} \
        --epochs {DETECTOR_EPOCHS} \
        --imgsz {DETECTOR_IMGSZ} \
        --batch {DETECTOR_BATCH} \
        --project "{DETECTOR_PROJECT}" \
        --name {DETECTOR_NAME}

Начинаю обучение с нуля.
Загружаю стартовые веса: yolo11s.pt
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/detection_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0,

### Если сессия оборвалась

Просто заново выполните ячейки этого ноутбука по порядку (`Runtime ->
Run all`). Шаг 5 найдёт уже закешированный датасет на Drive (сырые
данные повторно качать не придётся), а Шаг 6 сам обнаружит `last.pt` и
продолжит обучение с прерванной эпохи.

## Шаг 7. Экспорт в ONNX

Финальный шаг перед интеграцией в backend (`backend/app/services/detector.py`).
`--imgsz` здесь ДОЛЖЕН совпадать с тем, что использовалось при обучении
на Шаге 6.

In [14]:
best_checkpoint = os.path.join(DETECTOR_PROJECT, DETECTOR_NAME, 'weights', 'best.pt')
EXPORT_DIR = f'{DRIVE_PROJECT_ROOT}/detection_exported'

!python /content/ml/detection/export_detector.py \
    --checkpoint "{best_checkpoint}" \
    --output "{EXPORT_DIR}" \
    --imgsz {DETECTOR_IMGSZ}

print()
print(f'Готовая модель для backend: {EXPORT_DIR}/vehicle_detector.onnx')

Загружаю чекпоинт: /content/drive/MyDrive/VehicleReID/detection_runs/vehicle_detector/weights/best.pt
Экспортирую в ONNX (imgsz=640, opset=17)...
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/VehicleReID/detection_runs/vehicle_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (18.3 MB)
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 5 packages in 327ms
Prepared 1 package in 525ms
Installed 1 package in 7ms
 + onnxruntime==1.30.0

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates t

## Шаг 8. Быстрая визуальная проверка

Прогоняем несколько кадров из val-сплита через обученную модель и
смотрим на предсказанные боксы — до интеграции в backend полезно
убедиться "на глаз", что детектор находит машины разумно.

Полезно также сравнить с zero-shot: прогоните те же кадры через
необученный `YOLO('yolo11s.pt')` (с `classes=[2,5,7]` для car/bus/truck
из COCO) и визуально сравните результаты — это и есть тот самый
ablation "нужно ли вообще дообучение", о котором говорили.

In [15]:
from ultralytics import YOLO
import glob

model = YOLO(best_checkpoint)
val_images = glob.glob(f'{LOCAL_YOLO_DATASET}/images/val/*.jpg')[:6]

print('=== Дообученная модель ===')
results = model.predict(source=val_images, conf=0.4, save=False)
for r in results:
    r.show()

print('=== Для сравнения: zero-shot COCO-претрейн (без дообучения) ===')
zero_shot_model = YOLO(DETECTOR_MODEL)
# classes=[2,3,5,7] -> car, motorcycle, bus, truck в нумерации классов COCO
zero_shot_results = zero_shot_model.predict(source=val_images, conf=0.4, classes=[2, 3, 5, 7], save=False)
for r in zero_shot_results:
    r.show()

Output hidden; open in https://colab.research.google.com to view.

In [16]:
import os

for img_path in val_images:
    label_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    with open(label_path) as f:
        gt_count = sum(1 for line in f if line.strip())
    print(f'{os.path.basename(img_path)}: GT боксов = {gt_count}')

MVI_40243_img00851.jpg: GT боксов = 11
MVI_40141_img01061.jpg: GT боксов = 3
MVI_40243_img00101.jpg: GT боксов = 9
MVI_40243_img00686.jpg: GT боксов = 8
MVI_40243_img00251.jpg: GT боксов = 7
MVI_40141_img00351.jpg: GT боксов = 4
